# Getting Started with AkasicDB: Hybrid RAG

In this tutorial, you will use AkasicDB and open-source, text-based embeddings, and generative AI models to create a vector-graph hybrid RAG system.

## Prerequisite

* Modern Web Browser, such as Chrome, Firefox, Safari, and etc.
* Google Account (for Google Colab)
* **OpenAI API Key** (You can create one [here](https://platform.openai.com/api-keys).)
* **AkasicDB Playground DB instance** (You can create one [here](https://db.akasic.cloud).)

## Hybrid Retrieval-Augmented Generation (RAG) using AkasicDB

Retrieval-augmented generation (RAG) is a technique used with large language models (LLMs) to connect the model with a knowledge base of information outside the data the LLM has been trained on without having to perform fine-tuning.

To enhance retrieval accuracy and context, modern AI architectures are increasingly shifting toward **vector-graph hybrid RAG**. While traditional **vector-only RAG** excels at finding conceptually similar information through semantic search, it often struggles to connect the dots between complex entities and misses broader relational context.

Vector-graph hybrid RAG solves this by combining the semantic matching of vectors with the structural, relationship-driven mapping of knowledge graphs. To power this efficiently, organizations utilize **multi-model databases** that can natively store, index, and query both high-dimensional vectors and complex graph data within a single platform, delivering richer and more deeply contextualized LLM responses than vector search alone.

## Traditional, Text-based RAG

First, we build a simple text-based RAG system using AkasicDB. We will build a multi-modal RAG in the next tutorial.

## Step 0: Preparing environments

### ‼️**IMPORTANT**‼️ Setting up environemnt variables

For running this example, you **MUST** setup a public AkasicDB instance. You can create one from https://db.akasic.cloud.

You also need an [OpenAI API key](https://platform.openai.com/api-keys) for the chat and embedding models

Open your Google Colab notebook and click the key icon in the left-hand sidebar, then setup the following secrets:

- `DB_DSN`: the connection string copied from an AkasicDB instance created from https://db.akasic.cloud
- `OPENAI_API_KEY`: the OpenAI API key

If running outside Google Colab, export them as environment variables.

For local usages outside Google Colab, please refer to [the installation section](https://graphai-repository.github.io/akasicdb-docs/latest/en/installation/) of the AkasicDB documentation.


## Step 1: Install dependencies


In [ ]:
%pip install requests beautifulsoup4 transformers accelerate huggingface_hub akasicdb openai matplotlib networkx

Import required dependencies.

In [ ]:
import json
import os
from pathlib import Path
import textwrap
from typing import Iterable
import zlib

import argparse
import requests
from bs4 import BeautifulSoup

import numpy as np
import psycopg
from openai import OpenAI

Now checking for required secrets:

In [ ]:
def get_secret(name: str, default: str | None = None, *, required: bool = True) -> str | None:
    """
    Get a secret from Colab Secrets when running in Colab,
    otherwise from an environment variable.

    Args:
        name: Secret/env-var name, e.g. "OPENAI_API_KEY".
        default: Value to return if not found and required=False.
        required: Raise if missing.

    Returns:
        The secret value, or default/None if not required.
    """
    value = None

    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
    except Exception:
        # Not in Colab, or Colab secret not accessible.
        pass

    if value is None:
        value = os.getenv(name, default)

    if required and value is None:
        raise RuntimeError(
            f"Missing secret {name!r}. "
            f"In Colab, add it under Secrets and grant notebook access. "
            f"Outside Colab, set it as an environment variable."
        )

    return value

In [ ]:
DB_DSN = get_secret("DB_DSN")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY")

Here are some helper functions for later steps:

In [ ]:
def get_conn(conn_string: str=DB_DSN):
    return psycopg.connect(conn_string)

def get_openai_client(api_key=OPENAI_API_KEY):
    return OpenAI(api_key=api_key)

## Step 2: loading datasets

For this example, we use chunks and graphs extracted from [UltraDomain](https://huggingface.co/datasets/TommyChien/UltraDomain) dataset using [LightRAG](https://github.com/HKUDS/LightRAG).

Since graph extraction may take several hours, we use preprocessed data for 4 domains. For references, this is how the extraction process works:
```python
async def main(dataset: str = "agriculture"):
    # Initialize RAG instance
    rag = LightRAG(
        working_dir=WORKING_DIR,
        embedding_func=openai_embed,
        llm_model_func=gpt_4o_mini_complete,
    )
    # Initialize storage backends
    await rag.initialize_storages()

    # Insert data
    with open(f"./{dataset}_data/{dataset}.jsonl", "r", encoding="utf-8") as f:
        await rag.ainsert(f.read())

    # Export data
    await rag.aexport_data("{dataset}.csv", include_vector_data=True)
```


In [ ]:
import ipywidgets as widgets
from IPython.display import display

dataset_selector = widgets.Select(
    options=['agriculture', 'cs', 'legal', 'mix'],
    value='mix',
    description='Dataset:'
)

display(dataset_selector)


In [ ]:
dataset = dataset_selector.value

Now download and extract the selected dataset:

In [ ]:
PROCESSED_DATASET_URLS = {
    "agriculture": "https://graphai-public.s3.us-east-1.amazonaws.com/akasicdb-demo/lightrag_agriculture.zip",
    "cs": "https://graphai-public.s3.us-east-1.amazonaws.com/akasicdb-demo/lightrag_cs.zip",
    "legal": "https://graphai-public.s3.us-east-1.amazonaws.com/akasicdb-demo/lightrag_legal.zip",
    "mix": "https://graphai-public.s3.us-east-1.amazonaws.com/akasicdb-demo/lightrag_mix.zip",
}

def download_large_file(
    url: str,
    output_path: str | Path,
    chunk_size: int = 1024 * 1024,
    timeout: int = 30,
):
    """
    Download a large file using requests streaming mode.

    Args:
        url: File URL to download.
        output_path: Local file path to save the download.
        chunk_size: Number of bytes per chunk. Default is 1 MB.
        timeout: Request timeout in seconds.

    Returns:
        Path to the downloaded file.

    Raises:
        requests.HTTPError: If the HTTP request fails.
        requests.RequestException: For network-related errors.
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with requests.get(url, stream=True, timeout=timeout) as response:
        response.raise_for_status()

        with open(output_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:  # Filters out keep-alive chunks
                    file.write(chunk)

dataset_dir = Path('/content/datasets')
dataset_dir.mkdir(parents=True, exist_ok=True)

print(f"Downloading {dataset} datset into {str(dataset_dir)}...")

dataset_url = PROCESSED_DATASET_URLS[dataset]
!wget -c "{dataset_url}" -O /content/datasets/"{dataset}".zip

print("Extracting downloaded dataset...")

!unzip -o /content/datasets/"{dataset}".zip -d /content/datasets/

extracted_dir = dataset_dir / f"{dataset}_data"

print(f"Downloaded and extracted {dataset} into {extracted_dir}")

Now load the downloaded dataset into AkasicDB. This may take several minutes depending on the selected dataset.

In [ ]:
import base64
import zlib
import json
from pathlib import Path

import numpy as np
import psycopg


# ---------- vector decode ----------
def decode_vector(encoded: str):
    decoded = base64.b64decode(encoded)
    decompressed = zlib.decompress(decoded)
    vec_f16 = np.frombuffer(decompressed, dtype=np.float16)
    vector_f32 = vec_f16.astype(np.float32)

    return "[" + ",".join(str(x) for x in vector_f32) + "]"


# ---------- batching ----------
def iter_batches(items: list, batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]


def build_insert_params(kind: str, r: dict):
    raw_id = r["__id__"]
    vec = decode_vector(r["vector"])

    if kind == "chunks":
        return (
            raw_id,
            r["__created_at__"],
            r.get("content"),
            r.get("full_doc_id"),
            r.get("file_path"),
            vec,
        )

    elif kind == "entities":
        return (
            raw_id,
            r["__created_at__"],
            r.get("entity_name"),
            r.get("content"),
            r.get("source_id"),
            r.get("file_path"),
            vec,
        )

    elif kind == "relationships":
        return (
            raw_id,
            r["__created_at__"],
            r.get("src_id"),
            r.get("tgt_id"),
            r.get("source_id"),
            r.get("content"),
            r.get("file_path"),
            vec,
        )

    raise ValueError(f"Unsupported kind: {kind}")


# ---------- table creation ----------
def create_table_if_missing(cur: psycopg.Cursor, dataset: str, kind: str, dim: int):
    table = f"{dataset}_{kind}"

    if kind == "chunks":
        cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            id SERIAL PRIMARY KEY,
            raw_id TEXT UNIQUE,
            created_at BIGINT,
            content TEXT,
            full_doc_id TEXT,
            file_path TEXT,
            embedding VECTOR({dim})
        );
        """)

    elif kind == "entities":
        cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            id SERIAL PRIMARY KEY,
            raw_id TEXT UNIQUE,
            created_at BIGINT,
            entity_name TEXT,
            content TEXT,
            source_id TEXT,
            file_path TEXT,
            embedding VECTOR({dim})
        );
        """)

    elif kind == "relationships":
        cur.execute(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            id SERIAL PRIMARY KEY,
            raw_id TEXT UNIQUE,
            created_at BIGINT,
            src_entity TEXT,
            tgt_entity TEXT,
            source_id TEXT,
            content TEXT,
            file_path TEXT,
            embedding VECTOR({dim})
        );
        """)

    else:
        raise ValueError(f"Unsupported kind: {kind}")


# ---------- insert SQL ----------
def get_insert_sql(dataset: str, kind: str):
    table = f"{dataset}_{kind}"

    if kind == "chunks":
        return f"""
        INSERT INTO {table}
        (raw_id, created_at, content, full_doc_id, file_path, embedding)
        VALUES (%s, %s, %s, %s, %s, %s::vector)
        ON CONFLICT (raw_id) DO NOTHING;
        """

    elif kind == "entities":
        return f"""
        INSERT INTO {table}
        (raw_id, created_at, entity_name, content, source_id, file_path, embedding)
        VALUES (%s, %s, %s, %s, %s, %s, %s::vector)
        ON CONFLICT (raw_id) DO NOTHING;
        """

    elif kind == "relationships":
        return f"""
        INSERT INTO {table}
        (raw_id, created_at, src_entity, tgt_entity, source_id, content, file_path, embedding)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s::vector)
        ON CONFLICT (raw_id) DO NOTHING;
        """

    raise ValueError(f"Unsupported kind: {kind}")


# ---------- json loader ----------
def load_json(dataset: str, kind: str, data_dir: Path):
    path = data_dir / f"{dataset}_data/vdb_{kind}.json"

    with open(path, "r", encoding="utf-8") as f:
        j = json.load(f)

    return j["embedding_dim"], j["data"]


# ---------- load one kind ----------
def load_dataset_kind(
    cur: psycopg.Cursor,
    dataset: str,
    kind: str,
    data_dir: Path,
    batch_size: int = 1000,
):
    dim, rows = load_json(dataset, kind, data_dir)

    create_table_if_missing(cur, dataset, kind, dim)
    sql = get_insert_sql(dataset, kind)

    total = len(rows)
    processed = 0

    print(f"[{dataset}:{kind}] loading {total} rows in batches of {batch_size}...")

    for batch in iter_batches(rows, batch_size):
        params = [build_insert_params(kind, r) for r in batch]

        cur.executemany(sql, params)

        processed += len(batch)
        print(f"[{dataset}:{kind}] processed {processed}/{total} rows")

    print(f"[{dataset}:{kind}] done loading to a table")

    return dim


def load_dataset(
    dataset: str,
    cur: psycopg.Cursor,
    data_dir: Path,
    batch_size: int = 1000,
):
    dims = {}

    for kind in ["chunks", "entities", "relationships"]:
        dims[kind] = load_dataset_kind(
            cur,
            dataset,
            kind,
            data_dir,
            batch_size=batch_size,
        )

    return dims


def index_dataset(dataset: str, cur: psycopg.Cursor):
    graph_name = f"{dataset}_graph"

    def index_table(namespace: str | None, table: str):
        table_with_namespace = f"{namespace}.{table}" if namespace else table
        index_name = f"{namespace}_{table}_embedding_idx" if namespace else f"{table}_embedding_idx"

        print(f"[index] creating index on {table_with_namespace}.embedding ...")

        cur.execute(f"""
        CREATE INDEX IF NOT EXISTS {index_name}
        ON {table_with_namespace}
        USING vectoron (embedding vector_l2_ops);
        """)

    index_table(None, f"{dataset}_chunks")
    index_table(graph_name, "v_entity")
    index_table(graph_name, "e_relation")

    print(f"[index] all indexes created for dataset '{dataset}'")


def create_graph(
    dataset: str,
    cur: psycopg.Cursor,
    vertex_dim: int,
    edge_dim: int,
):
    graph_name = f"{dataset}_graph"
    vertex_label = "v_entity"
    edge_label = "e_relation"

    entities_table = f"{dataset}_entities"
    relationships_table = f"{dataset}_relationships"

    print(f"[graph] creating graph: {graph_name}")

    cur.execute(
        "SELECT akasicdb.define_graph(%s);",
        (graph_name,),
    )

    VERTEX_COLUMNS_WITH_TYPE = [
        "id INTEGER",
        "raw_id TEXT",
        "created_at BIGINT",
        "entity_name TEXT",
        "content TEXT",
        "source_id TEXT",
        "file_path TEXT",
        f"embedding VECTOR({vertex_dim})",
    ]

    vertex_query = f"""
        SELECT
            id,
            raw_id,
            created_at,
            entity_name,
            content,
            source_id,
            file_path,
            embedding
        FROM {entities_table}
    """

    cur.execute(
        """
        SELECT akasicdb.define_vertex(
            %s,           -- graph name
            %s,           -- vertex label
            %s,           -- columns with type
            %s,           -- table name
            %s            -- query
        );
        """,
        (
            graph_name,
            vertex_label,
            VERTEX_COLUMNS_WITH_TYPE,
            entities_table,
            vertex_query,
        ),
    )

    EDGE_COLUMNS_WITH_TYPE = [
        "created_at BIGINT",
        "source_id TEXT",
        "content TEXT",
        "file_path TEXT",
        f"embedding VECTOR({edge_dim})",
    ]

    edge_query = f"""
        SELECT
            rel.created_at,
            rel.source_id,
            rel.content,
            rel.file_path,
            rel.embedding
        FROM {entities_table} AS src,
             {relationships_table} AS rel,
             {entities_table} AS dst
        WHERE rel.src_entity = src.entity_name
          AND rel.tgt_entity = dst.entity_name
    """

    cur.execute(
        """
        SELECT akasicdb.define_edge(
            %s,           -- graph name
            %s,           -- edge label
            %s,           -- src vertex label
            %s,           -- dst vertex label
            %s,           -- columns with type
            %s,           -- query
            %s,           -- src table
            %s            -- dst table
        );
        """,
        (
            graph_name,
            edge_label,
            vertex_label,
            vertex_label,
            EDGE_COLUMNS_WITH_TYPE,
            edge_query,
            f"{entities_table} src",
            f"{entities_table} dst",
        ),
    )

    cur.execute(
        "SELECT akasicdb.create_graph(%s, replace_if_exists := true);",
        (graph_name,),
    )

    print(f"graph created: {graph_name}")


with get_conn() as conn:
    with conn.cursor() as cur:
        dims = load_dataset(dataset, cur, dataset_dir)
        create_graph(dataset, cur, vertex_dim=dims["entities"], edge_dim=dims["relationships"])
        index_dataset(dataset, cur)
    conn.commit()

Now verifying the graph has been loaded properly:

In [ ]:
from akasicdb.graph import execute_cypher

graph_name = f"{dataset}_graph"

rows = execute_cypher(
    DB_DSN,
    graph_name,
    """
    MATCH (n:v_entity)
    WITH n.vertex_id AS n_id
    LIMIT 10
    MATCH (n1:v_entity)-[r:e_relation]->(n2:v_entity)
    WHERE n1.vertex_id = n_id
    RETURN n1.entity_name AS src_name, r.content AS rel_content, n2.entity_name AS dst_name
    """,
)

for row in rows:
    print(row)

## Step 3: Processing user questions

For a user question, we will first extract keywords from it.

In [ ]:
KEYWORD_EXTRACTOR_PROMPT = """
# Role and Objective
You are an expert keyword extractor for a Retrieval-Augmented Generation (RAG) system. Analyze the user's query and identify both high-level and low-level keywords to support effective document retrieval.

# Goal
Extract two distinct types of keywords:
1. `high_level_keywords`: overarching concepts or themes that capture the user's core intent, subject area, or question type.
2. `low_level_keywords`: specific entities or details such as proper nouns, technical jargon, product names, or concrete items.

# Instructions
1. **Source of truth**: All keywords must be grounded in the user query. Prefer terms and phrases explicitly present in the query. Closely related phrases are allowed only when they directly reflect the query's stated topic or intent.
2. **Concise and meaningful**: Keywords should be concise words or meaningful phrases. Prioritize multi-word phrases when they represent a single concept. For example, from "latest financial report of Apple Inc.", extract "latest financial report" and "Apple Inc." rather than "latest", "financial", "report", and "Apple".
3. **Handle edge cases**: For queries that are too simple, vague, or nonsensical (for example, "hello", "ok", "asdfghjkl"), return a JSON object with empty lists for both keyword types.
4. **Content requirement**: For substantive queries, provide content in both `high_level_keywords` and `low_level_keywords`. If the query is too simple, vague, or nonsensical, both lists must be empty.
5. **Borderline queries**: Treat a query as vague only when it does not provide enough topical information to extract meaningful retrieval terms. Short but topical queries are substantive and should still produce keywords.
6. **Language**: All extracted keywords must be in the same language as the query. Proper nouns (for example, personal names, place names, organization names) should be kept in their original language.
7. **Ordering**: Order keywords within each list from most central and informative to less central and informative.

# Examples
## Example 1
Query: "How does international trade influence global economic stability?"

Output:
```json
{
  "high_level_keywords": ["International trade", "Global economic stability", "Economic impact"],
  "low_level_keywords": ["Trade agreements", "Tariffs", "Currency exchange", "Imports", "Exports"]
}
```

## Example 2
Query: "What are the environmental consequences of deforestation on biodiversity?"

Output:
```json
{
  "high_level_keywords": ["Environmental consequences", "Deforestation", "Biodiversity loss"],
  "low_level_keywords": ["Species extinction", "Habitat destruction", "Carbon emissions", "Rainforest", "Ecosystem"]
}
```

# Output Format
Return only a JSON object with exactly these two fields:
- `high_level_keywords`: an array of strings
- `low_level_keywords`: an array of strings

Do not include any text before or after the JSON object.

Schema:
```json
{
  "high_level_keywords": ["string"],
  "low_level_keywords": ["string"]
}
```

Edge-case output example:
```json
{
  "high_level_keywords": [],
  "low_level_keywords": []
}
```

# Verbosity
Be concise. Return only the required JSON object with no extra commentary.

# Stop Conditions
Finish once the JSON object has been produced in the required schema with exactly the two specified fields.
"""

def extract_keywords(question: str):
    response = get_openai_client().responses.create(
        model="gpt-5.4-nano",
        input=[
          {
            "role": "developer",
            "content": [
              {
                "type": "input_text",
                "text": KEYWORD_EXTRACTOR_PROMPT,
              }
            ]
          },
          {
            "role": "user",
            "content": [
              {
                "type": "input_text",
                "text": question,
              }
            ]
          },
        ],
        text={
          "format": {
            "type": "json_object"
          },
          "verbosity": "medium"
        },
        reasoning={
          "effort": "none",
        },
    )
    result_obj = json.loads(response.output_text)
    return result_obj["high_level_keywords"], result_obj["low_level_keywords"]

In [ ]:
SAMPLE_QUESTIONS = {
    "agriculture": [
        "What are the essential skills and knowledge needed for new beekeepers to succeed?",
        "Why is commitment important in beekeeping, and how can a beekeeper demonstrate it?",
    ],
    "cs": [
        "How does feature hashing compare to other term weighting schemes?",
        "How do the linear model and decision tree compare in terms of Mean Squared Error?",
    ],
    "legal": [
        "How do financial ratios align with the covenant requirements?",
        "How are synergies addressed in acquisition strategies?",
    ],
    "mix": [
        "How do emerging technologies accommodate growing urban populations as depicted?",
        "How do film techniques such as CGI and practical effects compare in historical representations?",
    ],
}

Embedding vectors for both the full question and keywords are generated as well:

In [ ]:
def generate_embedding(text: str):
    response = get_openai_client().embeddings.create(input=text, model="text-embedding-3-small")
    return response.data[0].embedding

Now we define functions for global queries and local queries:

In [ ]:
def global_search(high_level_embedding: list[float]):
    rows = execute_cypher(
        DB_DSN,
        graph_name,
        f"""
        MATCH (v:v_entity)
        WITH v.id AS entity_id
        ORDER BY v.embedding <-> '{high_level_embedding}'
        LIMIT 10
        MATCH (center:v_entity)-[r:e_relation]->(neighbor:v_entity)
        WHERE center.id = entity_id
        RETURN
            center.entity_name AS center,
            center.content AS center_content,
            center.source_id AS center_source_id,
            neighbor.entity_name AS neighbor,
            neighbor.content AS neighbor_content,
            neighbor.source_id AS neighbor_source_id
        """,
        params={
            "embedding": high_level_embedding
        }
    )
    return rows

def local_search(low_level_embedding: list[float]):
    rows = execute_cypher(
        DB_DSN,
        graph_name,
        f"""
        MATCH (s:v_entity)-[r:e_relation]->(t:v_entity)
        RETURN
            s.entity_name AS src_name,
            s.content AS src_content,
            s.source_id AS src_source_id,
            t.entity_name AS dst_name,
            t.content AS dst_content,
            t.source_id AS dst_source_id,
            r.source_id AS relation_source_id,
            r.content AS relation_content
        ORDER BY r.embedding <-> '{low_level_embedding}'
        LIMIT 10
        """,
        params={
            "embedding": low_level_embedding
        }
    )
    return rows

In [ ]:
def format_results(results):
    """Deduplicate results by source_id and concatenate unique contents."""
    seen_source_ids = set()
    seen_content = set()
    unique_contents = []

    for row in results:
        # Collect source IDs from all possible keys in global/local results
        source_ids = [
            row.get('center_source_id'),
            row.get('neighbor_source_id'),
            row.get('src_source_id'),
            row.get('dst_source_id'),
            row.get('relation_source_id')
        ]

        # Collect actual content text
        contents = [
            row.get('center_content'),
            row.get('neighbor_content'),
            row.get('src_content'),
            row.get('dst_content'),
            row.get('relation_content')
        ]

        for sid, content in zip(source_ids, contents):
            if content:
                # Check if we have already added this specific source_id or content string
                if (sid and sid not in seen_source_ids) or (not sid and content not in seen_content):
                    unique_contents.append(content)
                    if sid: seen_source_ids.add(sid)
                    seen_content.add(content)

    return "\n\n".join(unique_contents)

In [ ]:
def generate_response(context: str, question: str):
    response = get_openai_client().responses.create(
        model="gpt-5.4-nano",
        input=[
          {
            "role": "developer",
            "content": [
              {
                "type": "input_text",
                "text": f"Answer the question with the provided context below.\n\n <retrieved_context>\n\n{context}\n\n</retrieved_context>\nNever ask follow-up questions.",
              }
            ]
          },
          {
            "role": "user",
            "content": [
              {
                "type": "input_text",
                "text": question,
              }
            ]
          },
        ],
        text={
          "format": {
            "type": "text"
          },
          "verbosity": "medium"
        },
        reasoning={
          "effort": "none",
        },
    )

    return response.output[0].content[0].text


In [ ]:
for question in SAMPLE_QUESTIONS[dataset]:
    print(f"Q: \"{question}\"")
    high_level_keywords, low_level_keywords = extract_keywords(question)
    print(f"High-level keywords: {high_level_keywords}")
    print(f"Low-level keywords: {low_level_keywords}")

    question_embedding = generate_embedding(question)
    high_level_embedding = generate_embedding(", ".join(high_level_keywords))
    low_level_embedding = generate_embedding(", ".join(low_level_keywords))

    print("Retrieving context...")

    global_results = global_search(high_level_embedding)
    local_results = local_search(low_level_embedding)

    # Deduplicate and concatenate results
    combined_context = format_results(global_results + local_results)

    print(f"Context (first 500 characters):\n{combined_context[:500]}\n")

    # Generate Answer using OpenAI
    answer = generate_response(combined_context, question)

    print("-" * 40)
    print(f"A:\n{answer}")
    print("-" * 40)
    print()

### Visualize retrieved graph context

Instead of generating an answer, this step visualizes the `global_results` and `local_results` retrieved for each question.

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.lines import Line2D


def _short_label(value: str | None, max_chars: int = 28):
    text = str(value or "Unknown")
    return text if len(text) <= max_chars else text[: max_chars - 1] + "..."


def build_retrieval_graph(global_results: list[dict], local_results: list[dict]):
    retrieval_graph = nx.DiGraph()
    edge_modes: dict[tuple[str, str], set[str]] = {}

    def add_node(name: str | None, mode: str, source_id: str | None = None):
        if not name:
            return

        retrieval_graph.add_node(name)
        retrieval_graph.nodes[name].setdefault("modes", set()).add(mode)

        if source_id:
            retrieval_graph.nodes[name].setdefault("source_ids", set()).add(source_id)

    def add_edge(src: str | None, dst: str | None, mode: str):
        if not src or not dst:
            return

        edge_modes.setdefault((src, dst), set()).add(mode)

    for row in global_results:
        center = row.get("center")
        neighbor = row.get("neighbor")

        add_node(center, "global", row.get("center_source_id"))
        add_node(neighbor, "global", row.get("neighbor_source_id"))
        add_edge(center, neighbor, "global")

    for row in local_results:
        src = row.get("src_name")
        dst = row.get("dst_name")

        add_node(src, "local", row.get("src_source_id"))
        add_node(dst, "local", row.get("dst_source_id"))
        add_edge(src, dst, "local")

    for (src, dst), modes in edge_modes.items():
        label = "both" if modes == {"global", "local"} else next(iter(modes))
        retrieval_graph.add_edge(src, dst, modes=modes, label=label)

    return retrieval_graph


def visualize_retrieval_graph(global_results: list[dict], local_results: list[dict], title: str):
    retrieval_graph = build_retrieval_graph(global_results, local_results)

    if retrieval_graph.number_of_nodes() == 0:
        print("No retrieval results to visualize.")
        return

    node_count = retrieval_graph.number_of_nodes()
    fig_width = min(14, max(8, node_count * 0.9))
    fig_height = min(9, max(5, node_count * 0.55))
    pos = nx.spring_layout(retrieval_graph, seed=42, k=max(0.35, 1.2 / node_count ** 0.5))

    palette = {
        "global": "#4C78A8",
        "local": "#F58518",
        "both": "#54A24B",
    }

    node_colors = []
    for _, data in retrieval_graph.nodes(data=True):
        modes = data.get("modes", set())
        node_colors.append(palette["both"] if modes == {"global", "local"} else palette[next(iter(modes))])

    edge_colors = []
    edge_widths = []
    for src, dst, data in retrieval_graph.edges(data=True):
        modes = data.get("modes", set())
        mode = "both" if modes == {"global", "local"} else next(iter(modes))
        edge_colors.append(palette[mode])
        edge_widths.append(2.6 if mode == "both" else 1.8)

    plt.figure(figsize=(fig_width, fig_height))
    nx.draw_networkx_nodes(
        retrieval_graph,
        pos,
        node_color=node_colors,
        node_size=1300,
        alpha=0.92,
        edgecolors="#222222",
        linewidths=0.7,
    )
    nx.draw_networkx_edges(
        retrieval_graph,
        pos,
        edge_color=edge_colors,
        width=edge_widths,
        arrows=True,
        arrowsize=16,
        connectionstyle="arc3,rad=0.08",
    )
    nx.draw_networkx_labels(
        retrieval_graph,
        pos,
        labels={node: _short_label(node) for node in retrieval_graph.nodes},
        font_size=8,
        font_color="#111111",
        font_weight="bold",
        bbox={"boxstyle": "round,pad=0.18", "fc": "white", "ec": "none", "alpha": 0.72},
    )
    legend_handles = [
        Line2D([0], [0], marker="o", color="w", label="Global", markerfacecolor=palette["global"], markersize=10),
        Line2D([0], [0], marker="o", color="w", label="Local", markerfacecolor=palette["local"], markersize=10),
        Line2D([0], [0], marker="o", color="w", label="Both", markerfacecolor=palette["both"], markersize=10),
    ]
    plt.legend(handles=legend_handles, loc="best", frameon=False)
    plt.title(title, fontsize=12)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


for question in SAMPLE_QUESTIONS[dataset]:
    print(f"Q: \"{question}\"")
    high_level_keywords, low_level_keywords = extract_keywords(question)
    print(f"High-level keywords: {high_level_keywords}")
    print(f"Low-level keywords: {low_level_keywords}")

    high_level_embedding = generate_embedding(", ".join(high_level_keywords))
    low_level_embedding = generate_embedding(", ".join(low_level_keywords))

    print("Retrieving graph context...")
    global_results = global_search(high_level_embedding)
    local_results = local_search(low_level_embedding)
    print(f"Retrieved {len(global_results)} global rows and {len(local_results)} local rows.")

    visualize_retrieval_graph(
        global_results,
        local_results,
        title=f"Retrieved graph context: {_short_label(question, 70)}",
    )
    print("-" * 40)
